In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="1qMLQSmFWmPxaBH8uadN")
project = rf.workspace("sanny").project("iog-gender-age-cw4kg")
version = project.version(2)
dataset = version.download("folder")


loading Roboflow workspace...
loading Roboflow project...


In [ ]:
import os

dataset_path = dataset.location

print("Dataset location:", dataset_path)
print(os.listdir(dataset_path))

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:
        dirs[:] = []

Dataset location: /content/IOG-gender-age--2
['README.dataset.txt', 'train', 'test', 'valid', 'README.roboflow.txt']
IOG-gender-age--2/
    train/
        Male_3-18/
        Female_19-36/
        Female_3-18/
        Female_37-65/
        Male_37-65/
        Male_19-36/
    test/
        Male_3-18/
        Female_19-36/
        Female_3-18/
        Female_37-65/
        Male_37-65/
        Male_19-36/
    valid/
        Male_3-18/
        Female_19-36/
        Female_3-18/
        Female_37-65/
        Male_37-65/
        Male_19-36/


In [ ]:
import glob

csv_files = glob.glob(dataset_path + "/**/*.csv", recursive=True)

print("CSV files:")
for f in csv_files:
    print(f)

CSV files:


In [ ]:
import os
import shutil

original_dataset = dataset.location
converted_dataset = "/content/person_type_folder_dataset"

class_mapping = {
    "Female_19-36": "adult_female",
    "Male_19-36": "adult_male",
    "Female_3-18": "young_girl",
    "Male_3-18": "young_boy",
    "Female_37-65": "elderly_female",
    "Male_37-65": "elderly_male",
}

splits = ["train", "valid", "test"]

if os.path.exists(converted_dataset):
    shutil.rmtree(converted_dataset)

for split in splits:
    for old_class, new_class in class_mapping.items():
        old_path = os.path.join(original_dataset, split, old_class)
        new_path = os.path.join(converted_dataset, split, new_class)

        os.makedirs(new_path, exist_ok=True)

        if os.path.exists(old_path):
            for file in os.listdir(old_path):
                if file.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                    src = os.path.join(old_path, file)
                    dst = os.path.join(new_path, file)
                    shutil.copy(src, dst)

print("Converted dataset created:", converted_dataset)

Converted dataset created: /content/person_type_folder_dataset


In [ ]:
import os

dataset_path = "/content/person_type_folder_dataset"

train_dir = os.path.join(dataset_path, "train")
valid_dir = os.path.join(dataset_path, "valid")
test_dir = os.path.join(dataset_path, "test")

def count_images(folder):
    print(f"\nChecking: {folder}")
    for class_name in sorted(os.listdir(folder)):
        class_path = os.path.join(folder, class_name)

        if os.path.isdir(class_path):
            images = [
                f for f in os.listdir(class_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
            ]
            print(f"{class_name}: {len(images)} images")

count_images(train_dir)
count_images(valid_dir)
count_images(test_dir)


Checking: /content/person_type_folder_dataset/train
adult_female: 2378 images
adult_male: 2374 images
elderly_female: 2380 images
elderly_male: 2380 images
young_boy: 1472 images
young_girl: 1608 images

Checking: /content/person_type_folder_dataset/valid
adult_female: 339 images
adult_male: 340 images
elderly_female: 340 images
elderly_male: 340 images
young_boy: 210 images
young_girl: 229 images

Checking: /content/person_type_folder_dataset/test
adult_female: 169 images
adult_male: 170 images
elderly_female: 170 images
elderly_male: 170 images
young_boy: 105 images
young_girl: 115 images


In [ ]:
!pip install torch torchvision matplotlib scikit-learn pillow tqdm -q

In [ ]:
import os
import copy
import json
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import datasets, models, transforms

from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
image_size = 224
batch_size = 32

train_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

valid_test_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
valid_dataset = datasets.ImageFolder(valid_dir, transform=valid_test_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=valid_test_transforms)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Train images:", len(train_dataset))
print("Valid images:", len(valid_dataset))
print("Test images:", len(test_dataset))

Classes: ['adult_female', 'adult_male', 'elderly_female', 'elderly_male', 'young_boy', 'young_girl']
Number of classes: 6
Train images: 12592
Valid images: 1798
Test images: 899


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(device)

print("Using device:", device)
print("Number of classes:", num_classes)

Using device: cuda
Number of classes: 6


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.1
)

In [ ]:
def train_model(model, train_loader, valid_loader, criterion, optimizer, scheduler, num_epochs=20):
    best_model_weights = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    train_loss_history = []
    valid_loss_history = []
    train_acc_history = []
    valid_acc_history = []

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        print("-" * 40)

        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in tqdm(train_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        scheduler.step()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = running_corrects.double() / len(train_loader.dataset)

        model.eval()
        valid_running_loss = 0.0
        valid_running_corrects = 0

        with torch.no_grad():
            for inputs, labels in tqdm(valid_loader):
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                valid_running_loss += loss.item() * inputs.size(0)
                valid_running_corrects += torch.sum(preds == labels.data)

        epoch_valid_loss = valid_running_loss / len(valid_loader.dataset)
        epoch_valid_acc = valid_running_corrects.double() / len(valid_loader.dataset)

        train_loss_history.append(epoch_train_loss)
        valid_loss_history.append(epoch_valid_loss)
        train_acc_history.append(epoch_train_acc.item())
        valid_acc_history.append(epoch_valid_acc.item())

        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
        print(f"Valid Loss: {epoch_valid_loss:.4f} | Valid Acc: {epoch_valid_acc:.4f}")

        if epoch_valid_acc > best_acc:
            best_acc = epoch_valid_acc
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), "/content/best_person_type_classifier.pth")
            print("✅ Best model saved")

    model.load_state_dict(best_model_weights)

    return model, train_loss_history, valid_loss_history, train_acc_history, valid_acc_history

In [ ]:
model, train_loss, valid_loss, train_acc, valid_acc = train_model(
    model,
    train_loader,
    valid_loader,
    criterion,
    optimizer,
    scheduler,
    num_epochs=10
)


Epoch 1/10
----------------------------------------


100%|██████████| 57/57 [00:04<00:00, 13.57it/s]


Train Loss: 1.4404 | Train Acc: 0.4097
Valid Loss: 1.1573 | Valid Acc: 0.5412
✅ Best model saved

Epoch 2/10
----------------------------------------


100%|██████████| 57/57 [00:04<00:00, 14.10it/s]


Train Loss: 0.9711 | Train Acc: 0.6117
Valid Loss: 0.9819 | Valid Acc: 0.6029
✅ Best model saved

Epoch 3/10
----------------------------------------


...

Train Loss: 0.2661 | Train Acc: 0.9096
Valid Loss: 1.0807 | Valid Acc: 0.6357

Epoch 9/10
----------------------------------------


100%|██████████| 57/57 [00:03<00:00, 14.96it/s]


Train Loss: 0.2538 | Train Acc: 0.9111
Valid Loss: 1.0969 | Valid Acc: 0.6363

Epoch 10/10
----------------------------------------


100%|██████████| 57/57 [00:03<00:00, 14.37it/s]

Train Loss: 0.2394 | Train Acc: 0.9170
Valid Loss: 1.0985 | Valid Acc: 0.6363


In [ ]:
model.load_state_dict(torch.load("/content/best_person_type_classifier.pth", map_location=device))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

100%|██████████| 29/29 [00:03<00:00,  8.99it/s]

Classification Report:
                precision    recall  f1-score   support

  adult_female       0.62      0.56      0.59       169
    adult_male       0.69      0.68      0.68       170
elderly_female       0.65      0.71      0.68       170
  elderly_male       0.72      0.74      0.73       170
     young_boy       0.63      0.70      0.66       105
    young_girl       0.56      0.51      0.53       115

      accuracy                           0.65       899
     macro avg       0.65      0.65      0.65       899
  weighted avg       0.65      0.65      0.65       899



In [ ]:
final_model_path = "/content/person_type_classifier.pth"
class_names_path = "/content/person_type_classes.json"

torch.save(model.state_dict(), final_model_path)

with open(class_names_path, "w") as f:
    json.dump(class_names, f)

print("Model saved:", final_model_path)
print("Class names saved:", class_names_path)

Model saved: /content/person_type_classifier.pth
Class names saved: /content/person_type_classes.json


In [ ]:
from google.colab import files

files.download("/content/person_type_classifier.pth")
files.download("/content/person_type_classes.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>